# ConMedRL end-to-end clinical OCRL workflow

This notebook demonstrates the complete supported path:

1. preprocess MIMIC-IV, SICdb, or reviewed NWICU data;
2. inspect the dynamic state/action/constraint contract;
3. create ConMedRL train/validation/test loaders;
4. start FQI and objective/constraint FQE training;
5. update the Lagrange multiplier and evaluate on held-out test states;
6. optionally obtain d3rlpy and FHIR R4 outputs.

The defaults do **not** launch extraction or training. Set the environment path and the two run flags explicitly. The short training block is a connectivity demonstration, not a clinical result.

In [ ]:
from pathlib import Path

from ConMedRL.data import PreprocessConfig, build_dataset, validate_rl_contract
from ConMedRL.data.config import CohortConfig
from ConMedRL.data.profiles import load_nwicu_profile

# Choose: "mimic-iv", "sicdb", or "generic" (reviewed NWICU profile below).
DATABASE = "generic"
TASK = "extubation"  # or "discharge"
DATA_DIR = Path(r"C:\path\to\your\icu\data")
OUTPUT_DIR = Path("workflow_output")
RUN_PREPROCESS = False
RUN_SHORT_TRAIN = False
# Required before training: one clinically justified limit per constraint.
# Extubation uses ICU LOS hours, without scaling; e.g. [24.0] means 24 hours.
CONSTRAINT_LIMITS = None

# The built-in extubation task intentionally has one constraint only:
# remaining ICU length of stay in hours (con_cost_0).
if DATABASE == "generic":
    dataset_spec, task_spec, approval_hash = load_nwicu_profile(
        TASK,
        data_dir=DATA_DIR if RUN_PREPROCESS else None,
    )
    config = PreprocessConfig(
        database="generic",
        task=TASK,
        data_dir=DATA_DIR,
        output_dir=OUTPUT_DIR,
        dataset_spec=dataset_spec,
        task_spec=task_spec,
        approved_plan_hash=approval_hash,
        output_formats=("csv",),
    )
else:
    config = PreprocessConfig(
        database=DATABASE,
        task=TASK,
        data_dir=DATA_DIR,
        output_dir=OUTPUT_DIR,
        cohort=CohortConfig(
            decision_epoch_hours=12.0,
            los_threshold=15.0,
        ),
        output_formats=("csv",),
    )

bundle = build_dataset(config) if RUN_PREPROCESS else None
print("Configuration ready. Set RUN_PREPROCESS=True after reviewing paths/specs.")

In [ ]:
if bundle is not None:
    validate_rl_contract(bundle)
    summary = {
        "state_dim": bundle.state_dim,
        "state_variables": bundle.schema.names,
        "action": bundle.loader_action,
        "action_dim": bundle.action_dim,
        "num_constraints": bundle.num_constraints,
        "constraint_columns": [
            f"con_cost_{i}" for i in range(bundle.num_constraints)
        ],
        "split_rows": {
            name: len(tables.outcome)
            for name, tables in bundle.splits.items()
        },
        "written_files": bundle.written_files,
    }
    if TASK == "extubation":
        assert bundle.num_constraints == 1
        assert "con_cost_0" in bundle.train.outcome
        assert "con_cost_1" not in bundle.train.outcome
    summary
else:
    print("Preprocessing skipped.")

In [ ]:
from ConMedRL.conmedrl import RLConfig_custom
from ConMedRL.data_loader import TrainDataLoader, ValTestDataLoader

if bundle is not None:
    if RUN_SHORT_TRAIN and CONSTRAINT_LIMITS is None:
        raise ValueError(
            "Set CONSTRAINT_LIMITS explicitly before training; extubation LOS is in hours."
        )
    if CONSTRAINT_LIMITS is not None and len(CONSTRAINT_LIMITS) != bundle.num_constraints:
        raise ValueError(
            f"Expected {bundle.num_constraints} constraint limits, got {len(CONSTRAINT_LIMITS)}."
        )
    constraint_limits = (
        list(CONSTRAINT_LIMITS)
        if CONSTRAINT_LIMITS is not None
        else [0.0] * bundle.num_constraints  # unused while training is disabled
    )
    batch_size = min(64, len(bundle.train.outcome))
    rl_config = RLConfig_custom(
        algo_name="Constrained FQI + FQE",
        gamma=0.99,
        batch_size=batch_size,
        train_eps=1,
        train_eps_steps=2,
        weight_decay_fqi=1e-5,
        weight_decay_fqe=1e-5,
        optim_fqi="torch.optim.Adam",
        optim_fqe="torch.optim.Adam",
        loss_fqi="nn.MSELoss()",
        loss_fqe="nn.MSELoss()",
        memory_capacity=max(len(bundle.train.outcome), batch_size),
        target_update=1,
        tau=0.01,
        lr_fqi=1e-3,
        lr_fqe_obj=1e-3,
        lr_fqe_con_list=[1e-3] * bundle.num_constraints,
        lr_lambda_list=[1e-2] * bundle.num_constraints,
        constraint_num=bundle.num_constraints,
        threshold_list=constraint_limits,
        device_type="cuda",
        activation_function_fqi="relu",
        activation_params_fqi={},
        activation_function_fqe="relu",
        activation_params_fqe={},
        lambda_update="PG with bound",
        bound_lambda=10.0,
    )

    train_loader = TrainDataLoader(
        cfg=rl_config,
        **bundle.loader_kwargs("train"),
    )
    train_loader.data_buffer_train(
        action_name=bundle.loader_action,
        done_condition=None,
        num_constraint=bundle.num_constraints,
    )

    val_loader = ValTestDataLoader(
        cfg=rl_config,
        **bundle.loader_kwargs("val"),
    )
    val_loader.data_buffer(
        action_name=bundle.loader_action,
        done_condition=None,
        num_constraint=bundle.num_constraints,
    )
    print("ConMedRL replay buffers are ready.")
else:
    print("Run preprocessing before constructing RL loaders.")

In [ ]:
from ConMedRL.conmedrl import RLTraining

if bundle is not None and RUN_SHORT_TRAIN:
    trainer = RLTraining(
        cfg=rl_config,
        state_dim=bundle.state_dim,
        action_dim=bundle.action_dim,
        train_data_loader=train_loader.data_torch_loader_train,
        val_data_loader=val_loader.data_torch_loader,
    )
    fqi_agent = trainer.fqi_agent_config(hidden_layers=[64, 64], seed=7)
    fqe_objective = trainer.fqe_agent_config(
        eval_agent=fqi_agent,
        hidden_layers=[64, 64],
        eval_target="obj",
        seed=11,
    )
    fqe_constraints = [
        trainer.fqe_agent_config(
            eval_agent=fqi_agent,
            hidden_layers=[64, 64],
            eval_target=i,
            seed=12 + i,
        )
        for i in range(bundle.num_constraints)
    ]

    metrics = trainer.train(
        agent_fqi=fqi_agent,
        agent_fqe_obj=fqe_objective,
        agent_fqe_con_list=fqe_constraints,
        constraint=True,
        save_num=1,
        z_value=1.96,
    )
    print("Short FQI/FQE startup completed; this is not a converged model.")
else:
    print("Set RUN_SHORT_TRAIN=True only after preprocessing has completed.")

In [ ]:
if bundle is not None and RUN_SHORT_TRAIN:
    test_loader = ValTestDataLoader(
        cfg=rl_config,
        **bundle.loader_kwargs("test"),
    )
    test_loader.data_buffer(
        action_name=bundle.loader_action,
        done_condition=None,
        num_constraint=bundle.num_constraints,
    )
    test_states, *_ = test_loader.data_torch_loader(data_type="test")

    objective_estimate = fqe_objective.avg_Q_value_est(test_states, z_value=1.96)
    constraint_estimates = [
        agent.avg_Q_value_est(test_states, z_value=1.96)
        for agent in fqe_constraints
    ]
    held_out_fqe = {
        "objective_mean_upper_lower": objective_estimate,
        "constraint_mean_upper_lower": constraint_estimates,
        "final_lagrange_multipliers": {
            i: values[-1] for i, values in metrics[-1].items()
        },
    }
    held_out_fqe
else:
    print("Held-out FQE runs after the short training block.")

In [ ]:
BUILD_D3RLPY = False  # requires the optional d3rlpy dependency

if bundle is not None and BUILD_D3RLPY:
    mdp_bundle = bundle.to_mdp_dataset(
        split="train",
        negate_costs=True,
        include_constraints=True,
    )
    print("d3rlpy objective dataset:", type(mdp_bundle.objective).__name__)
    print("d3rlpy constraint datasets:", len(mdp_bundle.constraints))

if bundle is not None:
    print("CSV and requested export artifacts:", bundle.written_files)
else:
    print(
        "To export Parquet and de-identified FHIR R4 NDJSON during preprocessing, "
        "set output_formats=(\"csv\", \"parquet\", \"fhir\") in PreprocessConfig."
    )

## Interpretation and reproducibility checklist

- For **extubation**, `obj_cost` is the extubation-failure indicator and `num_constraints == 1`: `con_cost_0` is remaining ICU LOS in hours and is not scaled. A separate reintubation cost is intentionally not created because reintubation/extubation failure defines the objective.
- For **discharge**, the built-in task retains its two configured constraints.
- The state dimension is derived from the variables actually retained by preprocessing; never copy a historical hardcoded dimension.
- `CONSTRAINT_LIMITS` must be supplied explicitly in the same units as the corresponding costs; the notebook does not invent a default clinical threshold.
- FQE should be trained to convergence on training transitions before final policy evaluation. Use held-out test states only for the final locked analysis, not repeated model selection.
- Save the preprocessing configuration, bundle report, random seed, package version, and trained model states with each experiment.